To be converted to scripts eventually?

TODO:
get additional metadata from https://www.agathachristie.com/
- orig pub date
- work type
- detective
- alt titles?

Desired columns for LIB

- publisher ?
- title (from "title") 
- subject 1, subject 2, subject 3, (or more?), 
- se subjects
- series from "belongs-to-collection"?
- detective (add later) 
- collection type from "collection type"
- series position from "group-position"
- short description from "description"
- long description from "long-description"
- language from dc:language
- se word-count from "se:word-count"
- flesch scale from "se:reading-ease.flesch"
- source_url from "se:url.vcs.github"
- author from "author"
- author full name? from "se:name.person.full-name" (DON'T NEED)
- wikipedia url from "se:url.encyclopedia.wikipedia"
- number of chapters (DO LATER IN CORPUS)
- written_as (manually)
- work_type (manually)
- alt_titles (LATER) (manually)

Design decisions:
- choosing to treat each story in poirot investigates as its own document since the stories are the unit of composition (4/9/26)

## Raw Data Ingestion

The script below was run once to clone the corpus repositories from Standard Ebooks. 
It is preserved here for reproducibility. Do not re-run — raw_data/ is already populated.

```bash
#!/bin/bash
# Ingestion script for Agatha Christie Standard Ebooks Corpus

mkdir raw_data
cd raw_data

echo "Cloning repositories..."
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-murder-at-the-vicarage.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_giants-bread.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-secret-adversary.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-murder-on-the-links.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-man-in-the-brown-suit.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-big-four.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_poirot-investigates.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-mysterious-affair-at-styles.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-secret-of-chimneys.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-seven-dials-mystery.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-murder-of-roger-ackroyd.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-mystery-of-the-blue-train.git

echo "Ingestion complete."
```

## Construct LIB table

In [ ]:
# import libraries
import pandas as pd
import glob
import os
import re
import xml.etree.ElementTree as ET
from pathlib import Path
from bs4 import BeautifulSoup

In [ ]:
# # get list of all files in the raw_data folder
# root_dir = 'raw_data'
# file_list = []

# for root, dirs, files in os.walk(root_dir):
#     for file in files:
#         # Join the folder path and file name to get the full path
#         full_path = os.path.join(root, file)
#         file_list.append(full_path)

# file_list

In [ ]:
# helper functions for if a field is missing in the content.opf file, return None instead of throwing an error
def get_metadata(root, property_val, NS):
    el = root.find(f'opf:metadata/opf:meta[@property="{property_val}"]', NS)
    return el.text if el is not None else None

def get_dc(root, tag, NS):
    el = root.find(f'opf:metadata/dc:{tag}', NS)
    return el.text if el is not None else None

In [ ]:
# specify root directory (or hardcode?)
root_dir = 'raw_data'

# define namespaces used in StandardEbooks opf files (namespaces avoid tag collisions when multiple vocabularies are in one document)
NS = {
    'opf':  'http://www.idpf.org/2007/opf', # open packaging format
    'dc':   'http://purl.org/dc/elements/1.1/', # dublin core vocabulary (dublin core metadata terms)
    'rdf':  'http://www.w3.org/1999/02/22-rdf-syntax-ns#', # resource description framework
}

# get content.opf file paths
opf_files = sorted(Path(root_dir).glob('*/src/epub/content.opf'))

# initialize empty list to store book info and path
book_data = [] # a list of dictionaries, where each dictionary contains the book info for one book

for opf_path in opf_files:
    # get the book_id from the folder name
    book_id = opf_path.parts[1].replace('agatha-christie_', '') # .parts breaks a path into a tuple of components
    
    # initalize a list for subjects and se_subjects
    subjects = []
    se_subjects = []
    # get the tree and root for ElementTree parsing
    tree = ET.parse(opf_path)
    root = tree.getroot()

    # parse opf file and extract relevant fields
    publisher = get_dc(root, 'publisher', NS) # use helper function to guard against missing field
    title = get_dc(root, 'title', NS)
    series = get_metadata(root, 'belongs-to-collection', NS)
    col_type = get_metadata(root, 'collection-type', NS)
    series_pos = get_metadata(root, 'group-position', NS)
    short_desc = get_dc(root, 'description', NS)
    long_desc = get_metadata(root, 'se:long-description', NS)
    language = get_dc(root, 'language', NS)
    se_word_count = get_metadata(root, 'se:word-count', NS)
    se_reading_ease = get_metadata(root, 'se:reading-ease.flesch', NS)
    source_url = get_metadata(root, 'se:url.vcs.github', NS)
    author = get_dc(root, 'creator', NS)
    wikipedia_url = get_metadata(root, 'se:url.encyclopedia.wikipedia', NS)

    # for each subject in the content.opf file, append to subjects list
    subject_els = root.findall('dc:subject', NS)
    subjects = '|'.join([el.text for el in subject_els]) if subject_els else None # join and separate subjects with pipe and guard against missing field

    se_subject_els = root.findall('opf:metadata/opf:meta[@property="se:subject"]', NS)
    se_subjects = '|'.join([el.text for el in se_subject_els]) if se_subject_els else None

    # append extracted fields to book_data list as a dictionary
    book_data.append({
        'book_id': book_id,
        'publisher': publisher,
        'title': title,
        'author': author,
        'subjects': subjects,
        'se_subjects': se_subjects,
        'col_type': col_type,
        'series': series,
        'series_pos': series_pos,
        'short_desc': short_desc,
        'long_desc': long_desc,
        'language': language,
        'se_word_count': se_word_count,
        'se_reading_ease': se_reading_ease,
        'source_url': source_url,
        'wikipedia_url': wikipedia_url,
    })

# build dataframe from book_data list and set book_id as index
LIB = pd.DataFrame(book_data).set_index('book_id')
LIB

In [ ]:
# add manual fields
# add work type, writing_as, pub_year_original, and sleuth by hand (from agathachristie.com and wikipedia and the agatha christie app and the books themselves):
work_type = {
    "Giant's Bread": 'novel', # confirmed
    'Poirot Investigates': 'collection of short stories', # confirmed
    'The Big Four': 'novel', # confirmed
    'The Man in the Brown Suit': 'novel', # confirmed
    'The Murder at the Vicarage': 'novel', # confirmed
    'The Murder of Roger Ackroyd': 'novel', # confirmed
    'The Murder on the Links': 'novel', # confirmed
    'The Mysterious Affair at Styles': 'novel', # confirmed
    'The Mystery of the Blue Train': 'novel', # confirmed
    'The Secret Adversary': 'novel', # confirmed
    'The Secret of Chimneys': 'novel', # confirmed
    'The Seven Dials Mystery': 'novel' # confirmed
}

writing_as = {
    "Giant's Bread": 'Mary Westmacott', # confirmed
    'Poirot Investigates': 'Agatha Christie', # confirmed
    'The Big Four': 'Agatha Christie', # confirmed
    'The Man in the Brown Suit': 'Agatha Christie', # confirmed
    'The Murder at the Vicarage': 'Agatha Christie', # confirmed
    'The Murder of Roger Ackroyd': 'Agatha Christie', # confirmed
    'The Murder on the Links': 'Agatha Christie', # confirmed
    'The Mysterious Affair at Styles': 'Agatha Christie', # confirmed
    'The Mystery of the Blue Train': 'Agatha Christie', # confirmed
    'The Secret Adversary': 'Agatha Christie', # confirmed
    'The Secret of Chimneys': 'Agatha Christie', # confirmed
    'The Seven Dials Mystery': 'Agatha Christie' # confirmed
}

pub_year_original = {
    "Giant's Bread": 1930, # confirmed
    'Poirot Investigates': 1924, # confirmed
    'The Big Four': 1927, # confirmed
    'The Man in the Brown Suit': 1924, # confirmed
    'The Murder at the Vicarage': 1930, # confirmed
    'The Murder of Roger Ackroyd': 1926, # confirmed
    'The Murder on the Links': 1923, # confirmed
    'The Mysterious Affair at Styles': 1920, # confirmed
    'The Mystery of the Blue Train': 1928, # confirmed
    'The Secret Adversary': 1922, # confirmed
    'The Secret of Chimneys': 1925, # confirmed
    'The Seven Dials Mystery': 1929, # confirmed
}

sleuth = {
    "Giant's Bread": None, # confirmed
    'Poirot Investigates': "Hercule Poirot", # confirmed
    'The Big Four': "Hercule Poirot", # confirmed
    'The Man in the Brown Suit': "Colonel Race", # confirmed
    'The Murder at the Vicarage': "Miss Jane Marple", # confirmed
    'The Murder of Roger Ackroyd': "Hercule Poirot", # confirmed
    'The Murder on the Links': "Hercule Poirot", # confirmed
    'The Mysterious Affair at Styles': "Hercule Poirot", # confirmed
    'The Mystery of the Blue Train': "Hercule Poirot", # confirmed
    'The Secret Adversary': "Tommy and Tuppence", # confirmed
    'The Secret of Chimneys': "Superintendent Battle", # confirmed
    'The Seven Dials Mystery': "Superintendent Battle", # confirmed
}

In [ ]:
print(LIB.to_string())
print("\nMissing values:")
print(LIB.isnull().sum())

In [ ]:
# fix missingness

In [ ]:
# save to csv
# LIB.to_csv('LIB.csv', sep='|', index=True)
# print(LIB.to_string())